# NB_patient_360

Builds `<catalog>.gold.patient_360` as defined in `sttm_silver_to_gold.xlsx`, sheet `patient_360`.

* **Grain:** one row per patient
* **Driving table:** `<catalog>.silver.patients`
* **Joins (LEFT):** `silver.hospitals` on `registered_hospital_id = hospital_id`, `silver.insurance_providers` on `insurance_provider_id`, and `silver.lab_results` aggregated by `patient_id`
* **Filters:** none, so every patient is included
* **Load type:** FULL, using `CREATE OR REPLACE TABLE ... AS SELECT`

In [0]:
dbutils.widgets.text("catalog", "adb_caresync")
catalog = dbutils.widgets.get("catalog").strip()

src = f"{catalog}.silver"
target_table = f"{catalog}.gold.patient_360"
print(f"Source schema: {src} | Target: {target_table}")

In [0]:
# Columns each source table must have, per the STTM column mapping
required = {
    "patients": ["patient_id", "first_name", "last_name", "date_of_birth", "gender", "city", "state",
                 "registered_hospital_id", "insurance_provider_id"],
    "hospitals": ["hospital_id", "hospital_name"],
    "insurance_providers": ["insurance_provider_id", "provider_name"],
    "lab_results": ["lab_result_id", "patient_id", "result_status", "test_date"],
}

problems = []
for tbl, cols in required.items():
    fqn = f"{src}.{tbl}"
    if not spark.catalog.tableExists(fqn):
        problems.append(f"Missing table: {fqn}")
        continue
    existing = set(spark.table(fqn).columns)
    missing = [c for c in cols if c not in existing]
    if missing:
        problems.append(f"{fqn} is missing columns: {missing}")

if problems:
    raise Exception("Source validation failed:\n" + "\n".join(problems))
print("All source tables and columns present.")

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {target_table}
COMMENT 'Gold: one row per patient with hospital, insurance and lab test summary (STTM patient_360)'
AS
WITH lab_agg AS (
  SELECT
    patient_id,
    COUNT(*)                                AS total_tests,
    COUNT_IF(result_status = 'Abnormal')    AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')    AS critical_test_count,
    MAX(test_date)                          AS last_test_date
  FROM {src}.lab_results
  GROUP BY patient_id
)
SELECT
  CAST(p.patient_id AS INT)                                             AS patient_id,
  CAST(p.first_name AS STRING)                                          AS first_name,
  CAST(p.last_name AS STRING)                                           AS last_name,
  CAST(p.date_of_birth AS DATE)                                         AS date_of_birth,
  CAST(FLOOR(datediff(current_date(), p.date_of_birth) / 365.25) AS INT) AS age,
  CAST(p.gender AS STRING)                                              AS gender,
  CAST(p.city AS STRING)                                                AS city,
  CAST(p.state AS STRING)                                               AS state,
  CAST(p.registered_hospital_id AS INT)                                 AS hospital_id,
  CAST(h.hospital_name AS STRING)                                       AS hospital_name,
  CAST(p.insurance_provider_id AS INT)                                  AS insurance_provider_id,
  CAST(ip.provider_name AS STRING)                                      AS insurance_provider_name,
  CAST(COALESCE(l.total_tests, 0) AS INT)                               AS total_tests,
  CAST(COALESCE(l.abnormal_test_count, 0) AS INT)                       AS abnormal_test_count,
  CAST(COALESCE(l.critical_test_count, 0) AS INT)                       AS critical_test_count,
  CAST(l.last_test_date AS DATE)                                        AS last_test_date
FROM {src}.patients p
LEFT JOIN {src}.hospitals h
  ON p.registered_hospital_id = h.hospital_id
LEFT JOIN {src}.insurance_providers ip
  ON p.insurance_provider_id = ip.insurance_provider_id
LEFT JOIN lab_agg l
  ON p.patient_id = l.patient_id
""")
print(f"Rebuilt {target_table}")

In [0]:
record_count = spark.table(target_table).count()
print(f"{target_table} record count: {record_count}")
dbutils.notebook.exit(str(record_count))